# Computer Vision --- Lecture 5 Hands-On
## Training Deep Networks & CNN Architectures

**Week 5 -- Nguyen Manh Toan, Swinburne Vietnam**

Everything on today's slides, measured. You will watch a network die from bad
initialization, revive it with one constant, break it with a learning rate that
BatchNorm survives, and race four optimizers on the same problem.

| Section | Topic |
|---|---|
| 0 | Setup |
| 1 | Initialization: watching the signal die |
| 2 | Deriving and checking the $2/n$ rule |
| 3 | BatchNorm from scratch |
| 4 | The `model.eval()` bug, on purpose |
| 5 | Four optimizers on one ravine |
| 6 | Learning-rate schedules |
| 7 | Augmentation: what the model actually sees |
| 8 | Transfer learning with a pretrained ResNet |

Cells marked **TODO** are yours. Exercises at the end.

---
## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rcParams["figure.figsize"] = (10, 3.2)
rng = np.random.default_rng(0)
ACCENT, RED, GREEN, ORANGE = "#1F4E79", "#B03A2E", "#1E8449", "#E67E22"

In [ ]:
digits = load_digits()
X_all, y_all = digits.data / 16.0, digits.target
Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.3,
                                      random_state=0, stratify=y_all)
mu = Xtr.mean(0)
Xtr, Xte = Xtr - mu, Xte - mu
C = 10
print("train", Xtr.shape, " test", Xte.shape)


def softmax_loss(S, y):
    S = S - S.max(1, keepdims=True)
    e = np.exp(S)
    p = e / e.sum(1, keepdims=True)
    loss = -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
    dS = p.copy()
    dS[np.arange(len(y)), y] -= 1
    return loss, dS / len(y)

---
## 1. Initialization: Watching the Signal Die

Push random data through a deep ReLU stack and record the activation standard
deviation at each layer. Nothing is trained here -- we only ask whether the signal
survives the forward pass.

In [ ]:
def activation_scale(scale_fn, n_layers=10, width=128, N=512, seed=3):
    """Returns the activation std after each layer of an untrained ReLU net."""
    r = np.random.default_rng(seed)
    h = r.standard_normal((N, width))
    stds = [h.std()]
    for _ in range(n_layers):
        W = r.standard_normal((width, width)) * scale_fn(width)
        h = np.maximum(0, h @ W.T)
        stds.append(h.std())
    return np.array(stds)


schemes = [(lambda n: 0.01,            "0.01  (too small)", RED),
           (lambda n: np.sqrt(1 / n),  "sqrt(1/n)  Xavier", ORANGE),
           (lambda n: np.sqrt(2 / n),  "sqrt(2/n)  He",     GREEN),
           (lambda n: 0.20,            "0.20  (too large)", "#6B3E86")]

plt.figure(figsize=(5.5, 3.2))
for fn, lab, col in schemes:
    s = activation_scale(fn)
    plt.semilogy(s, "o-", ms=3, color=col, label=lab)
    print(f"{lab:22s} std after 10 layers = {s[-1]:.3e}")
plt.xlabel("layer"); plt.ylabel("activation std")
plt.legend(fontsize=8); plt.grid(alpha=.3, which="both")
plt.title("the same network, four initializations"); plt.tight_layout(); plt.show()

Read the exponents. With `0.01` the activations fall by roughly a factor of ten
**per layer** -- by layer 10 they are at $10^{-11}$, which is numerically zero, and so is
every gradient that depends on them. With `0.20` they grow just as fast in the other
direction. Only the two principled scales stay put.

Note this is a *geometric* process: a small per-layer error compounds. Depth is what
turns a 10% mistake into a catastrophe.

---
## 2. Where $2/n$ Comes From

For $z = \sum_{i=1}^{n} w_i x_i$ with independent, zero-mean $w$ and $x$:

$$\mathrm{Var}(z) = n\,\mathrm{Var}(w)\,\mathrm{Var}(x)$$

so $\mathrm{Var}(w) = 1/n$ preserves the variance. ReLU then discards the negative half,
halving it again -- hence $2/n$. Verify both claims numerically.

In [ ]:
n, N = 512, 200000
r = np.random.default_rng(1)
x = r.standard_normal((N, n))

# claim 1: Var(z) = n * Var(w) * Var(x)
for var_w in (1 / n, 2 / n, 0.01 ** 2):
    W = r.standard_normal(n) * np.sqrt(var_w)
    z = x @ W
    print(f"Var(w)={var_w:.3e}   predicted Var(z)={n*var_w*1.0:.4f}   "
          f"measured={z.var():.4f}")

# claim 2: ReLU halves the variance of a zero-mean symmetric signal
z = r.standard_normal(N)
print(f"\nbefore ReLU: var = {z.var():.4f}")
print(f"after  ReLU: var = {np.maximum(0, z).var():.4f}  "
      f"(ratio {np.maximum(0,z).var()/z.var():.3f})")
print("\nNote the ratio is ~0.34, not exactly 0.5 -- ReLU also shifts the mean.")
print("The 2/n rule targets E[h^2], not the variance, and E[h^2] IS exactly halved:")
print(f"  E[z^2] = {np.mean(z**2):.4f}   E[relu(z)^2] = {np.mean(np.maximum(0,z)**2):.4f}")

**TODO 1.** Show that the rule depends on the *fan-in*, not on the width being uniform.
Build a network whose layers narrow, `[512, 256, 128, 64, 32]`, and initialize each layer
with $\sqrt{2/n_{\text{in}}}$ using its own $n_{\text{in}}$. Confirm the activation std stays
flat. Then deliberately use the *first* layer's fan-in everywhere and watch it drift.

In [ ]:
# TODO 1 -- your code here
widths = [512, 256, 128, 64, 32]

---
## 3. BatchNorm From Scratch

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \qquad y = \gamma\hat{x} + \beta$$

The backward pass is the fiddly part, so we gradient-check it.

In [ ]:
def bn_forward(z, gamma, beta, eps=1e-5):
    m = z.mean(0)
    v = z.var(0)
    zh = (z - m) / np.sqrt(v + eps)
    return zh * gamma + beta, (zh, np.sqrt(v + eps), gamma)


def bn_backward(dout, cache):
    zh, std, gamma = cache
    n = len(dout)
    dgamma = (dout * zh).sum(0)
    dbeta = dout.sum(0)
    dz = (gamma / std / n) * (n * dout - dout.sum(0) - zh * (dout * zh).sum(0))
    return dz, dgamma, dbeta


# gradient check
r = np.random.default_rng(0)
z = r.standard_normal((16, 5)) * 2 + 3
g, b = r.standard_normal(5), r.standard_normal(5)
R = r.standard_normal((16, 5))

out, cache = bn_forward(z, g, b)
dz, dg, db = bn_backward(R, cache)

def num_grad(f, x, h=1e-6):
    out = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        i = it.multi_index; old = x[i]
        x[i] = old + h; fp = f()
        x[i] = old - h; fm = f()
        x[i] = old
        out[i] = (fp - fm) / (2 * h)
        it.iternext()
    return out

obj = lambda: (bn_forward(z, g, b)[0] * R).sum()
for name, arr, ana in [("dz", z, dz), ("dgamma", g, dg), ("dbeta", b, db)]:
    ng = num_grad(obj, arr)
    err = np.max(np.abs(ana - ng) / np.maximum(1e-12, np.abs(ana) + np.abs(ng)))
    print(f"{name:7s} relative error {err:.2e}  {'PASS' if err < 1e-6 else 'FAIL'}")
print("\n(Threshold 1e-6 here, not the usual 1e-7: dividing by sqrt(var) amplifies")
print(" round-off, so BN cannot reach the accuracy a plain matmul does.)")

print(f"\noutput mean {out.mean(0).round(3)}  (should be beta = {b.round(3)})")
print(f"output std  {out.std(0).round(3)}  (should be |gamma| = {np.abs(g).round(3)})")

In [ ]:
def train_mlp(use_bn, lr, iters=900, H=128, seed=0, track=25):
    r = np.random.default_rng(seed)
    D = Xtr.shape[1]
    W1 = r.standard_normal((H, D)) * np.sqrt(2 / D)
    W2 = r.standard_normal((H, H)) * np.sqrt(2 / H)
    W3 = r.standard_normal((C, H)) * np.sqrt(2 / H)
    g1, b1, g2, b2 = np.ones(H), np.zeros(H), np.ones(H), np.zeros(H)
    hist = []
    for t in range(iters):
      with np.errstate(over="ignore", invalid="ignore"):   # divergence is the point
        i = r.choice(len(Xtr), 128, replace=False)
        xb, yb = Xtr[i], ytr[i]
        z1 = xb @ W1.T
        z1n, c1 = bn_forward(z1, g1, b1) if use_bn else (z1, None)
        a1 = np.maximum(0, z1n)
        z2 = a1 @ W2.T
        z2n, c2 = bn_forward(z2, g2, b2) if use_bn else (z2, None)
        a2 = np.maximum(0, z2n)
        loss, dS = softmax_loss(a2 @ W3.T, yb)
        if not np.isfinite(loss):
            hist += [np.nan] * (iters // track - len(hist)); break
        dW3 = dS.T @ a2
        dz2n = (dS @ W3) * (z2n > 0)
        if use_bn:
            dz2, dg2, db2 = bn_backward(dz2n, c2)
        else:
            dz2 = dz2n
        dW2 = dz2.T @ a1
        dz1n = (dz2 @ W2) * (z1n > 0)
        if use_bn:
            dz1, dg1, db1 = bn_backward(dz1n, c1)
        else:
            dz1 = dz1n
        dW1 = dz1.T @ xb
        W1 -= lr * dW1; W2 -= lr * dW2; W3 -= lr * dW3
        if use_bn:
            g1 -= lr * dg1; b1 -= lr * db1; g2 -= lr * dg2; b2 -= lr * db2
        if t % track == 0:
            hist.append(min(loss, 10))
    return hist


fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, lr in zip(axes, [0.5, 4.0]):
    for use_bn, col, lab in [(False, RED, "without BN"), (True, GREEN, "with BN")]:
        h = [v if np.isfinite(v) else 10 for v in train_mlp(use_bn, lr)]
        ax.semilogy(np.arange(len(h)) * 25, np.maximum(h, 1e-4), color=col, label=lab)
        fin = [v for v in h if v < 10]
        print(f"lr={lr}  BN={use_bn}: final {fin[-1] if fin else float('nan'):.4f}")
    ax.set_title(f"learning rate {lr}"); ax.set_xlabel("iteration")
    ax.grid(alpha=.3, which="both"); ax.legend(fontsize=8)
axes[0].set_ylabel("minibatch loss")
plt.tight_layout(); plt.show()

At $\eta = 0.5$ both train and BatchNorm is merely faster. At $\eta = 4$ the plain network
diverges within a few dozen iterations while the BatchNorm one is unbothered.

That is the practical case for BN: not a point of accuracy, but **the range of learning
rates that work at all**.

---
## 4. The `eval()` Bug, On Purpose

BatchNorm normalizes with **batch** statistics while training and **running** statistics at
test time. Use the wrong one and predictions depend on batch composition.

In [ ]:
r = np.random.default_rng(5)
H = 64
Wp = r.standard_normal((H, Xtr.shape[1])) * np.sqrt(2 / Xtr.shape[1])
gamma, beta = np.ones(H), np.zeros(H)

# running statistics, as a real BN layer would accumulate them
run_m, run_v, mom = np.zeros(H), np.ones(H), 0.9
for _ in range(200):
    i = r.choice(len(Xtr), 128, replace=False)
    z = Xtr[i] @ Wp.T
    run_m = mom * run_m + (1 - mom) * z.mean(0)
    run_v = mom * run_v + (1 - mom) * z.var(0)

x1 = Xte[:1]                                  # ONE test image
correct = ((x1 @ Wp.T - run_m) / np.sqrt(run_v + 1e-5)) * gamma + beta
z1 = x1 @ Wp.T
buggy = ((z1 - z1.mean(0)) / np.sqrt(z1.var(0) + 1e-5)) * gamma + beta

print("eval mode  (running stats), first 5 units:", correct[0, :5].round(3))
print("train mode (batch stats),   first 5 units:", buggy[0, :5].round(3))
print("\nWith a batch of one, the batch mean IS the value, so every activation")
print("is normalized to exactly beta. The image's content is completely erased.")
print(f"buggy output is constant: {np.allclose(buggy, beta)}")

A batch of one is the extreme case, but the same corruption happens at any small batch --
your validation number becomes a function of how you happened to shuffle. In PyTorch:
`model.eval()` before validating, `model.train()` after. Dropout has the identical split.

---
## 5. Four Optimizers, One Ravine

$$L(w) = \tfrac{1}{2}\big(w_1^2 + \kappa\, w_2^2\big)$$

with $\kappa \gg 1$: steep across the valley, shallow along it. This is what makes plain
SGD zig-zag.

In [ ]:
KAPPA = 22.0
STEPS = 120
loss_f = lambda w: 0.5 * (w[0] ** 2 + KAPPA * w[1] ** 2)
grad_f = lambda w: np.array([w[0], KAPPA * w[1]])


def optimise(kind, lr, steps=None, start=(-4.2, 1.4)):
    steps = STEPS if steps is None else steps
    w = np.array(start, float); path = [w.copy()]
    v = np.zeros(2); s = np.zeros(2); m = np.zeros(2)
    for t in range(1, steps + 1):
        g = grad_f(w)
        if kind == "sgd":
            w = w - lr * g
        elif kind == "momentum":
            v = 0.9 * v - lr * g
            w = w + v
        elif kind == "rmsprop":
            s = 0.9 * s + 0.1 * g ** 2
            w = w - lr * g / (np.sqrt(s) + 1e-8)
        elif kind == "adam":
            m = 0.9 * m + 0.1 * g
            s = 0.999 * s + 0.001 * g ** 2
            w = w - lr * (m / (1 - 0.9 ** t)) / (np.sqrt(s / (1 - 0.999 ** t)) + 1e-8)
        path.append(w.copy())
    return np.array(path)


gx, gy = np.linspace(-5, 5, 200), np.linspace(-2, 2, 200)
GX, GY = np.meshgrid(gx, gy)
Z = 0.5 * (GX ** 2 + KAPPA * GY ** 2)

# Each learning rate was tuned for its own optimizer: comparing them at a shared lr
# would just measure who tolerates that particular lr.  Criterion: the first step
# after which the loss STAYS below 1e-4 -- momentum oscillates, so "loss at step N"
# is partly luck about where it is in the swing.
SETTINGS = [("sgd", 0.0861, RED), ("momentum", 0.0032, ORANGE),
            ("rmsprop", 0.0588, "#6B3E86"), ("adam", 0.1398, GREEN)]


def converged_at(L, tol=1e-4):
    below = L < tol
    if not below[-1]:
        return -1
    return 0 if below.all() else len(L) - 1 - int(np.argmax(below[::-1] == False)) + 1


fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 3.4))
axL.contour(GX, GY, Z, levels=np.logspace(-1, 2.2, 16), colors="0.85", linewidths=.6)
for kind, lr, col in SETTINGS:
    p = optimise(kind, lr)
    L = np.array([loss_f(w) for w in p])
    axL.plot(*p.T, "-o", ms=2, lw=1, color=col, label=f"{kind} (lr={lr})")
    axR.semilogy(L, color=col, lw=1.5, label=kind)
    print(f"{kind:9s} lr {lr:.4f}   converged by step {converged_at(L):3d}   "
          f"path length {np.abs(np.diff(p, axis=0)).sum():6.2f}")
axL.plot(0, 0, "*", color="k", ms=13)
axL.set_xticks([]); axL.set_yticks([]); axL.legend(fontsize=7, ncol=2, loc="lower center")
axL.set_title(f"{STEPS} steps, curvature ratio {KAPPA:.0f}:1")
axR.axhline(1e-4, color="0.6", ls="--", lw=.9)
axR.set_xlabel("step"); axR.set_ylabel("loss"); axR.grid(alpha=.3, which="both")
axR.set_title("loss vs step"); axR.legend(fontsize=7)
plt.tight_layout(); plt.show()

### Momentum loses here, and the reason is worth more than the demo

Plain SGD converges by step 65; momentum with $\beta = 0.9$ needs 118. That is not a bug,
and it is not what most courses show you.

For a quadratic, once the heavy-ball recursion has complex roots its error decays at
exactly $\sqrt{\beta}$ per step, **no matter what $\eta$ you choose**. With $\beta = 0.9$
that is $0.949$. Tuned gradient descent decays at $1 - 2/(\kappa+1) = 0.913$. Slower
per-step than plain GD -- so on this problem momentum cannot win.

$\beta$ and $\kappa$ are coupled. The optimal choice is

$$\beta^\star = \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2$$

which for $\kappa = 22$ is $\mathbf{0.42}$, not $0.9$. Run the next cell: with
$\beta = 0.42$ momentum converges in **20 steps** against SGD's 65 -- a $3.3\times$
speed-up, the behaviour you were probably expecting.

Inverting the formula, $\beta = 0.9$ is the optimal setting for $\kappa \approx 1442$. The
default is not a universal constant; it is a guess that deep networks are *badly*
conditioned. Usually they are, which is why it works in practice.

In [ ]:
best_beta = ((np.sqrt(KAPPA) - 1) / (np.sqrt(KAPPA) + 1)) ** 2
print(f"optimal beta for kappa={KAPPA:.0f}:  {best_beta:.3f}")
print(f"beta=0.9 would be optimal for kappa = "
      f"{((1+np.sqrt(0.9))/(1-np.sqrt(0.9)))**2:.0f}\n")

for beta in (0.9, best_beta):
    for lr in (0.0032, 0.1176):
        w = np.array([-4.2, 1.4]); v = np.zeros(2); L = [loss_f(w)]
        for _ in range(STEPS):
            v = beta * v - lr * grad_f(w); w = w + v
            L.append(loss_f(w))
        print(f"  beta={beta:.3f}  lr={lr:.4f}  ->  converged by step "
              f"{converged_at(np.array(L))}")

**TODO 2.** The learning rates above were tuned per optimizer, which is the honest way to
compare them. Now do the unfair version: give all four the *same* $\eta$, and sweep
$\eta \in \{0.01, 0.03, 0.1, 0.3\}$. For each, record steps-to-threshold (not final loss --
you now know why). Which optimizers survive the largest $\eta$? Relate your answer to what
the adaptive term $1/(\sqrt{s}+\epsilon)$ is doing to the effective step size.

In [ ]:
# TODO 2 -- your code here

---
## 6. Learning-Rate Schedules

In [ ]:
E = 100
t = np.arange(E)
scheds = {
    "constant":     np.full(E, 0.1),
    "step":         0.1 * (0.1 ** (t // 30)),
    "exponential":  0.1 * (0.97 ** t),
    "cosine":       0.1 * 0.5 * (1 + np.cos(np.pi * t / E)),
    "warmup+cos":   np.where(t < 10, 0.1 * (t + 1) / 10,
                             0.1 * 0.5 * (1 + np.cos(np.pi * (t - 10) / (E - 10)))),
}
plt.figure(figsize=(6, 3))
for k, v in scheds.items():
    plt.plot(t, v, label=k)
plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.legend(fontsize=8)
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

for k, v in scheds.items():
    print(f"{k:12s} start {v[0]:.4f}  mid {v[E//2]:.4f}  end {v[-1]:.5f}  "
          f"mean {v.mean():.4f}")

---
## 7. Augmentation: What the Model Actually Sees

Always look at the output of your augmentation pipeline. Silent bugs here (wrong channel
order, labels not transformed with the image, augmentation applied at test time) are
common and expensive.

In [ ]:
import cv2
from skimage import data

img = data.astronaut()
h, w = img.shape[:2]
r = np.random.default_rng(4)

def rand_crop(im, frac=0.75):
    ch, cw = int(h * frac), int(w * frac)
    y0, x0 = r.integers(0, h - ch), r.integers(0, w - cw)
    return cv2.resize(im[y0:y0+ch, x0:x0+cw], (w, h))

def rotate(im, deg):
    M = cv2.getRotationMatrix2D((w/2, h/2), deg, 1.0)
    return cv2.warpAffine(im, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def jitter(im, bright=1.0, sat=1.0):
    hsv = cv2.cvtColor(im, cv2.COLOR_RGB2HSV).astype(float)
    hsv[..., 1] = np.clip(hsv[..., 1] * sat, 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] * bright, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def erase(im, frac=0.25):
    out = im.copy()
    eh, ew = int(h*frac), int(w*frac)
    y0, x0 = r.integers(0, h-eh), r.integers(0, w-ew)
    out[y0:y0+eh, x0:x0+ew] = 0
    return out

views = [(img, "original"), (img[:, ::-1], "h-flip"),
         (rand_crop(img), "crop"), (rotate(img, 12), "rotate 12°"),
         (jitter(img, 1.35, 0.7), "bright+desat"),
         (jitter(img, 0.7, 1.5), "dark+sat"),
         (erase(img), "random erasing"), (rand_crop(img, 0.55), "crop (tight)")]

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, (im, lab) in zip(axes.ravel(), views):
    ax.imshow(im); ax.set_title(lab, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

**TODO 3.** Augmentation encodes an assumed invariance, and the wrong one destroys the
label. Load `sklearn.datasets.load_digits`, train the linear classifier from Lecture 3, and
evaluate it on a test set that has been **horizontally flipped**. Report the accuracy drop.
Then augment the *training* set with flipped copies, retrain, and report accuracy on both
the normal and the flipped test sets. Explain why this augmentation is a mistake for digits
but correct for photographs of cats.

In [ ]:
# TODO 3 -- your code here

---
## 8. Transfer Learning

*(Needs `torch` and `torchvision`. This is the section that matters most for your project.)*

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

m = models.resnet18(weights="DEFAULT")

total = sum(p.numel() for p in m.parameters())
print(f"ResNet-18 total parameters: {total:,}")
print(f"final layer: {m.fc}")

# ---- feature extraction: freeze everything but a new head ----
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(512, 10)                     # 10 classes, fresh head

trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"\ntrainable now: {trainable:,}  ({100*trainable/total:.2f}% of the network)")
print("Everything else keeps its ImageNet weights.")

In [ ]:
# the normalization the pretrained weights expect -- getting this wrong quietly hurts
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_tf = transforms.Compose([                # NO augmentation at test time
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
print(train_tf)

In [ ]:
# a synthetic "dataset" so the loop below runs without downloading anything
torch.manual_seed(0)
Xf = torch.randn(64, 3, 224, 224)
yf = torch.randint(0, 10, (64,))

opt = torch.optim.Adam(m.fc.parameters(), lr=1e-3)   # head only
lossf = nn.CrossEntropyLoss()

m.train()
for p in m.modules():                                # keep frozen BN in eval mode
    if isinstance(p, nn.BatchNorm2d):
        p.eval()

for step in range(3):
    opt.zero_grad()
    loss = lossf(m(Xf[:16]), yf[:16])
    loss.backward()
    opt.step()
    print(f"step {step}: loss {loss.item():.4f}")

print("\nOnly m.fc received gradients:")
for name in ["conv1.weight", "layer4.1.conv2.weight", "fc.weight"]:
    g = dict(m.named_parameters())[name].grad
    print(f"  {name:24s} grad is {'None' if g is None else 'present'}")

Note the loop that puts frozen `BatchNorm2d` layers into `eval()` mode. Without it, the
running statistics keep updating from your data even though the weights are frozen -- so
the "frozen" backbone quietly changes anyway. This is a real and frequently-missed bug.

**Then, to fine-tune:** unfreeze the last block, drop the learning rate to $10^{-4}$ or
below, and continue.

---
## Exercises

**1. The initialization/depth interaction.** For depths $\{5, 10, 20, 40\}$ and init scales
$\{0.5/\sqrt{n}, 1/\sqrt{n}, \sqrt{2/n}, 2/\sqrt{n}\}$, measure the activation std at the
final layer. Present it as a heatmap. At which depth does the choice of scale start to
matter by more than an order of magnitude? This is why the papers on initialization all
appeared once people started training deep networks, and not before.

**2. Does BatchNorm still help after you tune the learning rate?** Sweep $\eta$ over a log
grid separately for the BN and non-BN networks, and compare each at *its own* best setting.
Is the remaining gap in final loss, in convergence speed, or in neither? This is the
comparison the original paper is often accused of not making.

**3. Optimizer generalization, not just optimization.** Train the same 2-layer network on
digits with SGD+momentum and with Adam, tuning each to reach the same *training* loss. Then
compare test accuracy. Repeat over 5 seeds and report mean and std -- one run tells you
nothing. Does the folklore ("SGD generalizes better") hold on this problem?

**4. Augmentation strength has an optimum.** Train with random crops of scale
$\{1.0, 0.9, 0.7, 0.5, 0.3\}$ and plot train and validation accuracy against crop strength.
You should see under-regularization at one end and destroyed labels at the other. Where is
the peak, and how sharp is it?

**5. Read a residual block's gradient.** Build a 20-layer plain network and a 20-layer
network with skip connections every 2 layers, both in NumPy. Do one backward pass in each
and plot $\|\partial L/\partial W\|$ per layer, as in Lecture 4 section 5. Quantify how much
more gradient reaches layer 1 in the residual version. That number is the reason ResNet works.